In [16]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.svm import SVR
import xgboost as xgb
import deepchem as dc
from deepchem.feat import RDKitDescriptors
from deepchem.models.torch_models import MLPModel
import matplotlib.pyplot as plt

def load_data_csv(train_file, test_file):
    # Load CSV files with columns "smiles" and "pKa"
    train_df = pd.read_csv(train_file)
    test_df = pd.read_csv(test_file)
    return train_df, test_df

def featurize_data(df, featurizer, smiles_col='smiles', target_col='pKa'):
    # Featurize molecules using RDKit descriptors (via DeepChem)
    smiles = df[smiles_col].tolist()
    y = df[target_col].values
    X = featurizer.featurize(smiles)
    return X, y

def scale_features(X_train, X_test):
    # Standardize features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled, scaler

def train_svm_model(X_train, y_train, X_test, y_test):
    # Train an SVM regressor (using RBF kernel)
    svm = SVR(kernel='rbf', C=1.0, epsilon=0.1)
    svm.fit(X_train, y_train)
    preds = svm.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    print("SVM RMSE: {:.2f}, R2: {:.2f}".format(rmse, r2))
    return svm, preds

def train_xgb_model(X_train, y_train, X_test, y_test):
    # Train an XGBoost regressor
    xgb_model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100,
                                 learning_rate=0.1, random_state=42)
    xgb_model.fit(X_train, y_train)
    preds = xgb_model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    print("XGBoost RMSE: {:.2f}, R2: {:.2f}".format(rmse, r2))
    return xgb_model, preds

def train_deepchem_mlp(train_dataset, test_dataset, n_features, nb_epoch=100, batch_size=32):
    # Train a deep neural network using DeepChem's PyTorch-based MLPModel.
    # The architecture has three hidden layers with 256 nodes each and dropout (0.25).
    model = MLPModel(n_tasks=1,
                     n_features=n_features,
                     layer_sizes=[256, 256, 256],
                     dropout=0.25,
                     learning_rate=0.001,
                     batch_size=batch_size,
                     model_dir='mlp_model',
                     mode='regression')
    model.fit(train_dataset, nb_epoch=nb_epoch)
    preds = model.predict(test_dataset)
    preds = preds.flatten()
    y_true = test_dataset.y.flatten()
    rmse = np.sqrt(mean_squared_error(y_true, preds))
    r2 = r2_score(y_true, preds)
    print("DeepChem MLP RMSE: {:.2f}, R2: {:.2f}".format(rmse, r2))
    return model, preds

def plot_predictions(y_true, preds, title):
    plt.figure(figsize=(6,6))
    plt.scatter(y_true, preds, alpha=0.6)
    plt.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--')
    plt.xlabel("Experimental pKa")
    plt.ylabel("Predicted pKa")
    plt.title(title)
    plt.show()

def main():
    # Specify file paths for Option 1 acidic pKa data
    train_file = "Opt1_acidic_tr.csv"
    test_file = "Opt1_acidic_tst.csv"
    
    # Load the CSV data
    train_df, test_df = load_data_csv(train_file, test_file)
    
    # Initialize the RDKit descriptors featurizer from DeepChem
    featurizer = RDKitDescriptors()
    
    # Featurize training and test data (assumes presence of "smiles" and "pKa" columns)
    X_train, y_train = featurize_data(train_df, featurizer, smiles_col='smiles', target_col='pKa')
    X_test, y_test = featurize_data(test_df, featurizer, smiles_col='smiles', target_col='pKa')
    
    # Scale features
    X_train_scaled, X_test_scaled, scaler = scale_features(X_train, X_test)
    
    # --- Model 1: SVM ---
    svm_model, svm_preds = train_svm_model(X_train_scaled, y_train, X_test_scaled, y_test)
    plot_predictions(y_test, svm_preds, "SVM Predictions")
    
    # --- Model 2: XGBoost ---
    xgb_model, xgb_preds = train_xgb_model(X_train_scaled, y_train, X_test_scaled, y_test)
    plot_predictions(y_test, xgb_preds, "XGBoost Predictions")
    
    # --- Model 3: DeepChem MLP (PyTorch-based) ---
    # Create DeepChem NumpyDatasets from the scaled features
    train_dataset = dc.data.NumpyDataset(X_train_scaled, y_train)
    test_dataset = dc.data.NumpyDataset(X_test_scaled, y_test)
    
    n_features = X_train_scaled.shape[1]
    mlp_model, mlp_preds = train_deepchem_mlp(train_dataset, test_dataset, n_features, nb_epoch=100, batch_size=32)
    plot_predictions(y_test, mlp_preds, "DeepChem MLP Predictions")

if __name__ == "__main__":
    main()


ImportError: cannot import name 'MLPModel' from 'deepchem.models.torch_models' (C:\Users\Fahad\anaconda3\envs\vs_comparator\lib\site-packages\deepchem\models\torch_models\__init__.py)

In [15]:
!pip install deepchem.models.torch_models

ERROR: Could not find a version that satisfies the requirement deepchem.models.torch_models (from versions: none)
ERROR: No matching distribution found for deepchem.models.torch_models
